## Device Specifications

| Property        | Value                  |
|----------------|------------------------|
| GPU            | NVIDIA L40S            |
| VRAM           | 48 GB GDDR6            |
| Compute        | SM 8.9 (Ada Lovelace)  |
| BF16 Support   | Native                 |
| FP8 Support    | Native (SM 8.9+)       |
| TF32 Support   | Native                 |
| NF4 (4-bit)    | Supported              |
| torch.compile  | Supported (Inductor)   |

## 1 · Install dependencies

In [ ]:
import subprocess, sys

pkgs = [
    "transformers", "datasets", "scikit-learn",
    "pandas", "numpy", "torch", "torchao",
    "bitsandbytes", "optimum", "quanto"
]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
print("All packages ready.")


## 2 · Imports & seeds

In [ ]:
import os, gc, time, warnings, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    sm_major  = torch.cuda.get_device_properties(0).major
    sm_minor  = torch.cuda.get_device_properties(0).minor
    print(f"GPU    : {gpu_name}")
    print(f"VRAM   : {total_mem:.1f} GB")
    print(f"Compute: SM {sm_major}.{sm_minor}")
    print(f"Expected: NVIDIA L40S | 48 GB GDDR6 | SM 8.9")
else:
    print("GPU    : Not available — running on CPU")
    print("Expected: NVIDIA L40S | 48 GB GDDR6 | SM 8.9")


## 3 · Config

In [ ]:
MODEL_NAME      = "google/flan-t5-base"
BATCH_SIZE      = 64
FINETUNE_EPOCHS = 2
RECOVERY_EPOCHS = 1
RECOVERY_SIZE   = 5000
GRAD_ACCUM      = 1
CLIP_NORM       = 1.0
LR              = 5e-5
LR_RECOVERY     = 1e-4
SPARSITY        = 0.20
MAX_INPUT_LEN   = 256
MAX_TARGET_LEN  = 16
CALIB_SIZE      = 512
ARTIFACTS_DIR   = "artifacts-flan-pruning"
METRICS_CSV     = "pruning_metrics_sheet4.csv"
FINETUNED_PATH  = "artifacts-flan-pruning/flan-t5-base_finetuned.pt"

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print(f"Model   : {MODEL_NAME}")
print(f"Sparsity: {SPARSITY:.0%}  |  Base FT epochs: {FINETUNE_EPOCHS}  |  Recovery: {RECOVERY_EPOCHS} epoch / {RECOVERY_SIZE} samples")
print(f"Batch   : {BATCH_SIZE}  |  LR: {LR}  |  LR_recovery: {LR_RECOVERY}")
print(f"TF32    : {torch.backends.cuda.matmul.allow_tf32}")


## 4 · Data — AG News via HuggingFace (`sh0416/ag_news`)

In [ ]:
print("Loading ag_news from HuggingFace …")
hf_data = load_dataset("sh0416/ag_news")
print(hf_data)
print("\nSample:", hf_data["train"][0])


In [ ]:
LABEL_MAP = {1: "world", 2: "sports", 3: "business", 4: "sci/tech"}

def hf_to_seq2seq(split_data):
    rows = []
    for item in split_data:
        snippet = (str(item["title"]) + " - " + str(item["description"]))[:300]
        prompt  = (
            "Classify the following news article into one of four categories: "
            "world, sports, business, or sci/tech. "
            f"Article: '{snippet}'. Predict category:"
        )
        rows.append({
            "input_text":  prompt,
            "target_text": LABEL_MAP[item["label"]],
        })
    return pd.DataFrame(rows)

print("Formatting train split …")
full_train_df = hf_to_seq2seq(hf_data["train"])

print("Formatting test split …")
test_df = hf_to_seq2seq(hf_data["test"])

train_df, val_df = train_test_split(
    full_train_df, test_size=0.10,
    stratify=full_train_df["target_text"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
print("Label distribution (train):")
print(train_df["target_text"].value_counts())


## 5 · Dataset class & DataLoaders

In [ ]:
class AGNewsSeq2SeqDataset(Dataset):
    def __init__(self, dataframe, tokenizer,
                 max_input=MAX_INPUT_LEN, max_target=MAX_TARGET_LEN):
        self.data       = dataframe
        self.tokenizer  = tokenizer
        self.max_input  = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        enc = self.tokenizer(
            row["input_text"],
            max_length=self.max_input, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        dec = self.tokenizer(
            row["target_text"],
            max_length=self.max_target, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        label_ids = dec.input_ids.squeeze()
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids":      enc.input_ids.squeeze(),
            "attention_mask": enc.attention_mask.squeeze(),
            "labels":         label_ids,
        }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded — vocab size: {tokenizer.vocab_size:,}")


In [ ]:
calib_df    = train_df.iloc[:CALIB_SIZE].reset_index(drop=True)
recovery_df = train_df.sample(RECOVERY_SIZE, random_state=42).reset_index(drop=True)

NUM_WORKERS = 4

calib_loader    = DataLoader(AGNewsSeq2SeqDataset(calib_df,    tokenizer),
                             batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
recovery_loader = DataLoader(AGNewsSeq2SeqDataset(recovery_df, tokenizer),
                             batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
train_loader    = DataLoader(AGNewsSeq2SeqDataset(train_df,    tokenizer),
                             batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
test_loader     = DataLoader(AGNewsSeq2SeqDataset(test_df,     tokenizer),
                             batch_size=32,
                             num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

print(f"Calib    : {len(calib_loader)} batches")
print(f"Recovery : {len(recovery_loader)} batches  ({RECOVERY_SIZE} samples)")
print(f"Train    : {len(train_loader)} batches")
print(f"Test     : {len(test_loader)} batches")


## 6 · Helper utilities

In [ ]:
def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def model_size_mb(model, path=None):
    if path and os.path.exists(path):
        return os.path.getsize(path) / (1024 ** 2)
    return sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)

def get_prunable_params(model):
    return [(n, p) for n, p in model.named_parameters()
            if "weight" in n and p.dim() > 1]


## 7 · Training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, device, epoch_idx=0):
    model.train()
    optimizer.zero_grad()
    for i, batch in enumerate(loader):
        ids  = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        lbls = batch["labels"].to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            loss = model(input_ids=ids, attention_mask=mask, labels=lbls).loss / GRAD_ACCUM

        loss.backward()
        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            optimizer.step()
            optimizer.zero_grad()
        if i % 50 == 0:
            print(f"  [epoch {epoch_idx+1}] batch {i:4d} | loss {loss.item()*GRAD_ACCUM:.4f}")


## 8 · Evaluation

In [ ]:
def evaluate(model, loader, tokenizer, device, save_path=None):
    model.eval()
    label_map  = {"world": 0, "sports": 1, "business": 2, "sci/tech": 3}
    preds, gts = [], []
    total_time = 0.0

    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            lbls = batch["labels"]

            t0  = time.time()
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = model.generate(
                    input_ids=ids,
                    attention_mask=mask,
                    max_new_tokens=5,
                    pad_token_id=tokenizer.pad_token_id,
                )
            total_time += time.time() - t0

            for j in range(ids.size(0)):
                gen   = tokenizer.decode(out[j], skip_special_tokens=True).strip().lower()
                valid = lbls[j][lbls[j] != -100]
                gt    = tokenizer.decode(valid, skip_special_tokens=True).strip().lower()
                preds.append(gen)
                gts.append(gt)

    n      = len(preds)
    y_true = [label_map.get(g, 0) for g in gts]
    y_pred = [label_map.get(p, 0) for p in preds]
    return {
        "Accuracy"  : round(accuracy_score(y_true, y_pred) * 100, 2),
        "Macro-F1"  : round(f1_score(y_true, y_pred, average="macro") * 100, 2),
        "Latency"   : round(total_time / n, 4) if n else 0,
        "Size (MB)" : round(model_size_mb(model, save_path), 2),
    }


## 8.5 · Base Fine-Tune

Fine-tune the raw pretrained model on AG News **before** any pruning.  
This is the key change for 90%+ post-pruning accuracy: prune from a strong base, not random pretrain weights.

In [ ]:
import os

def run_base_finetune(model_name=MODEL_NAME, epochs=FINETUNE_EPOCHS, lr=LR,
                      save_path=FINETUNED_PATH, force_retrain=False):
    if os.path.exists(save_path) and not force_retrain:
        print(f"  [Base FT] Found checkpoint at {save_path} — loading instead of retraining.")
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16,
            device_map="cuda" if torch.cuda.is_available() else "auto",
        )
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))
        model.eval()
        return model

    print(f"  [Base FT] Fine-tuning {model_name} for {epochs} epoch(s) …")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name, torch_dtype=torch.bfloat16,
        device_map="cuda" if torch.cuda.is_available() else "auto",
    )
    compiled  = torch.compile(model)
    use_fused = torch.cuda.is_available()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, fused=use_fused)

    for ep in range(epochs):
        print(f"  Epoch {ep+1}/{epochs}")
        train_one_epoch(compiled, train_loader, optimizer, DEVICE, epoch_idx=ep)

    torch.save(model.state_dict(), save_path)
    print(f"  [Base FT] Saved fine-tuned checkpoint → {save_path}")
    model.eval()
    return model

print("Running base fine-tune …")
base_model = run_base_finetune()

print("\nEvaluating fine-tuned base model …")
base_metrics = evaluate(base_model, test_loader, tokenizer, DEVICE, FINETUNED_PATH)
base_metrics["Method"]     = "FINETUNED_BASE"
base_metrics["Model Name"] = MODEL_NAME.split("/")[-1]
print(f"  → Acc {base_metrics['Accuracy']:.2f}%  |  F1 {base_metrics['Macro-F1']:.2f}%  |  "
      f"Latency {base_metrics['Latency']:.4f}s  |  Size {base_metrics['Size (MB)']:.1f} MB")

del base_model
clear_gpu()

## 9 · Pruning Methods

### 9a · Magnitude Pruning

In [ ]:
def magnitude_prune(model, sparsity):
    print(f"  [Magnitude] sparsity={sparsity:.0%}")
    with torch.no_grad():
        for name, param in get_prunable_params(model):
            flat      = param.data.abs().view(-1)
            k         = int(sparsity * flat.numel())
            if k == 0:
                continue
            threshold = torch.kthvalue(flat, k).values
            param.data.mul_(param.data.abs() > threshold)
    return model


### 9b · Movement Pruning

In [ ]:
def movement_prune(model, calib_loader, sparsity, device):
    print(f"  [Movement] collecting gradients …")
    model.train()
    model.zero_grad()
    for batch in calib_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)
        model(input_ids=ids, attention_mask=mask, labels=lbls).loss.backward()

    print(f"  [Movement] applying mask at sparsity={sparsity:.0%}")
    with torch.no_grad():
        for name, param in get_prunable_params(model):
            if param.grad is None:
                continue
            scores    = (param.data * param.grad).abs()
            k         = int(sparsity * scores.numel())
            if k == 0:
                continue
            threshold = torch.kthvalue(scores.view(-1), k).values
            param.data.mul_(scores > threshold)
    model.zero_grad()
    return model


### 9c · SparseGPT

In [ ]:
class SparseGPTLayer:

    def __init__(self, layer):
        self.layer     = layer
        self.n_cols    = layer.weight.shape[1]
        self.H         = torch.zeros(self.n_cols, self.n_cols,
                                     device=layer.weight.device, dtype=torch.float32)
        self.n_samples = 0

    def add_batch(self, inp):
        x = inp.reshape(-1, self.n_cols).float()
        self.H        += x.T @ x
        self.n_samples += x.size(0)

    def prune(self, sparsity, block_size=128):
        if self.n_samples == 0:
            return
        W = self.layer.weight.data.clone().float()
        H = self.H / self.n_samples
        H.diagonal().add_(0.01 * H.diagonal().mean())

        try:
            H_inv = torch.cholesky_inverse(torch.linalg.cholesky(H))
        except torch.linalg.LinAlgError:
            H_inv = torch.diag(1.0 / H.diagonal().clamp(min=1e-8))

        mask = torch.zeros_like(W, dtype=torch.bool)
        for start in range(0, self.n_cols, block_size):
            end    = min(start + block_size, self.n_cols)
            W_blk  = W[:, start:end].clone()
            H_blk  = H_inv[start:end, start:end]
            h_diag = H_blk.diagonal().clamp(min=1e-8)

            n_prune = int(sparsity * (end - start))
            if n_prune == 0:
                continue
            scores   = W_blk ** 2 / h_diag.unsqueeze(0)
            thresh   = torch.kthvalue(scores.reshape(-1), n_prune).values
            blk_mask = scores <= thresh
            mask[:, start:end] = blk_mask

            survive = (~blk_mask).float()
            err     = (W_blk * blk_mask.float()) / h_diag.unsqueeze(0)
            W[:, start:end] -= (err @ H_blk) * survive

        W[mask] = 0.0
        self.layer.weight.data = W.to(self.layer.weight.dtype)


def sparsegpt_prune(model, calib_loader, sparsity, device):
    print(f"  [SparseGPT] registering hooks …")
    sg_layers, hooks = {}, {}

    def make_hook(sg):
        def hook(module, inp, out):
            sg.add_batch(inp[0].detach())
        return hook

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            sg = SparseGPTLayer(module)
            sg_layers[name] = sg
            hooks[name]     = module.register_forward_hook(make_hook(sg))

    print(f"  [SparseGPT] {len(sg_layers)} Linear layers | calibration pass …")
    model.eval()
    with torch.no_grad():
        for batch in calib_loader:
            model(input_ids=batch["input_ids"].to(device),
                  attention_mask=batch["attention_mask"].to(device),
                  labels=batch["labels"].to(device))

    for h in hooks.values():
        h.remove()

    print(f"  [SparseGPT] pruning at sparsity={sparsity:.0%} …")
    for sg in sg_layers.values():
        sg.prune(sparsity)
    return model

### 9d · HAWQ → bitsandbytes INT8

`bitsandbytes` loads the model directly in INT8 using LLM.int8() quantization.  
It performs **layer-wise sensitivity-aware** quantization — outlier channels stay in FP16,  
remaining weights quantized to INT8. Works natively with Flan-T5 via HuggingFace.


In [ ]:
def hawq_bnb_quantize(model_name, device):
    from transformers import BitsAndBytesConfig
    import bitsandbytes

    print(f"  [bitsandbytes] loading {model_name} in NF4 (L40S SM 8.9 supports 4-bit) …")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.eval()
    print(f"  [bitsandbytes] NF4 model ready.")
    return model


### 9e · ZeroQuant → quanto INT8

`quanto` (by HuggingFace) quantizes weights and activations to INT8/INT4.  
Better accuracy retention than bitsandbytes at INT4, full Flan-T5 support,  
and actively maintained as of 2024.


In [ ]:
def zeroquant_quanto_quantize(model, device):
    from quanto import quantize, freeze
    try:
        from quanto import qfloat8
        wtype = qfloat8
        print(f"  [quanto] using FP8 (SM 8.9 native) …")
    except ImportError:
        from quanto import qint8
        wtype = qint8
        print(f"  [quanto] qfloat8 unavailable — falling back to INT8 …")

    quantize(model, weights=wtype)
    freeze(model)
    model = model.to(device)
    print(f"  [quanto] quantization complete.")
    return model

## 10 · Per-method pipeline

In [ ]:
METHOD_REGISTRY = {
    "magnitude" : {"fn": magnitude_prune,          "type": "prune",        "needs_calib": False},
    "movement"  : {"fn": movement_prune,            "type": "prune",        "needs_calib": True},
    "sparsegpt" : {"fn": sparsegpt_prune,           "type": "prune",        "needs_calib": True},
    "hawq"      : {"fn": hawq_bnb_quantize,         "type": "quant_bnb",    "needs_calib": False},
    "zeroquant" : {"fn": zeroquant_quanto_quantize,  "type": "quant_quanto", "needs_calib": False},
}


def run_method(method_name, model_name=MODEL_NAME):
    print(f"\n{'='*60}")
    print(f" Method : {method_name.upper()}")
    print(f" Model  : {model_name}  |  Sparsity: {SPARSITY:.0%}")
    print(f"{'='*60}")

    entry       = METHOD_REGISTRY[method_name]
    fn          = entry["fn"]
    mtype       = entry["type"]
    needs_calib = entry["needs_calib"]

    if mtype == "quant_bnb":
        model = fn(model_name, DEVICE)

    elif mtype == "quant_quanto":
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16,
            device_map="cuda" if torch.cuda.is_available() else "auto",
        )
        if os.path.exists(FINETUNED_PATH):
            model.load_state_dict(torch.load(FINETUNED_PATH, map_location=DEVICE))
            print(f"  Loaded fine-tuned weights from {FINETUNED_PATH}")
        model = fn(model, DEVICE)

    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16,
            device_map="cuda" if torch.cuda.is_available() else "auto",
        )
        if os.path.exists(FINETUNED_PATH):
            model.load_state_dict(torch.load(FINETUNED_PATH, map_location=DEVICE))
            print(f"  Loaded fine-tuned weights from {FINETUNED_PATH}")
        else:
            print(f"  Fine-tuned checkpoint not found — pruning from raw pretrain weights")

        if needs_calib:
            model = fn(model, calib_loader, SPARSITY, DEVICE)
        else:
            model = fn(model, SPARSITY)

        print(f"  Recovery fine-tune: {RECOVERY_EPOCHS} epoch(s) on {RECOVERY_SIZE} samples  (LR={LR_RECOVERY})")
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR_RECOVERY, fused=torch.cuda.is_available())
        for ep in range(RECOVERY_EPOCHS):
            train_one_epoch(model, recovery_loader, optimizer, DEVICE, epoch_idx=ep)

    short = model_name.split("/")[-1]
    path  = os.path.join(ARTIFACTS_DIR, f"{short}_{method_name}.pt")
    try:
        torch.save(model.state_dict(), path)
        print(f"  Saved → {path}")
    except Exception as e:
        print(f"  Save skipped ({e})")
        path = None

    print("  Evaluating …")
    metrics = evaluate(model, test_loader, tokenizer, DEVICE, path)
    metrics["Method"]     = method_name.upper()
    metrics["Model Name"] = short

    print(f"  → Acc {metrics['Accuracy']:.2f}%  |  "
          f"F1 {metrics['Macro-F1']:.2f}%  |  "
          f"Latency {metrics['Latency']:.4f}s  |  "
          f"Size {metrics['Size (MB)']:.1f} MB")

    del model
    clear_gpu()
    return metrics

## 11 · Run all methods

In [ ]:
METHODS_TO_RUN = ["magnitude", "movement", "sparsegpt", "hawq", "zeroquant"]

all_metrics = []
for method in METHODS_TO_RUN:
    result = run_method(method, MODEL_NAME)
    if result:
        all_metrics.append(result)

print("\nAll methods complete.")


## 12 · Results — Sheet4 format

In [ ]:
if all_metrics:
    base_row = [base_metrics] if 'base_metrics' in dir() else []
    all_rows = base_row + all_metrics
    df = pd.DataFrame(all_rows)[["Method", "Model Name", "Accuracy", "Macro-F1", "Latency", "Size (MB)"]]
    display(df)
    df.to_csv(os.path.join(ARTIFACTS_DIR, METRICS_CSV), index=False)
    print(f"\nSaved → {os.path.join(ARTIFACTS_DIR, METRICS_CSV)}")

## 13 · Optional — sweep all Flan-T5 sizes


In [ ]:
ALL_MODELS = [
    "google/flan-t5-small",
    "google/flan-t5-base",
    "google/flan-t5-large",
    "google/flan-t5-xl",
    "google/flan-t5-xxl",
]
METHODS_MULTI = ["magnitude", "movement", "sparsegpt", "hawq", "zeroquant"]

multi_metrics = []
for mname in ALL_MODELS:
    tok = AutoTokenizer.from_pretrained(mname)
    calib_loader = DataLoader(AGNewsSeq2SeqDataset(calib_df, tok),
                              batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    train_loader = DataLoader(AGNewsSeq2SeqDataset(train_df, tok),
                              batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(AGNewsSeq2SeqDataset(test_df, tok),
                              batch_size=32,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    tokenizer = tok

    for method in METHODS_MULTI:
        res = run_method(method, mname)
        if res:
            multi_metrics.append(res)

if multi_metrics:
    mdf = pd.DataFrame(multi_metrics)[[
        "Method", "Model Name", "Accuracy", "Macro-F1", "Latency", "Size (MB)"
    ]]
    mdf.to_csv(os.path.join(ARTIFACTS_DIR, "pruning_all_models.csv"), index=False)
    display(mdf)